In [1]:
import numpy, pandas, sklearn.preprocessing, sklearn.neighbors, pickle

In [2]:
dataframe = pandas.read_csv("data/serbia_optical_internet_plans.csv")
dataframe

,snapshot_date,provider,plan_name,download_mbps,upload_mbps,symmetric,bundle_type,price_rsd_month,price_rsd_no_contract,contract_months,technology,notes,source_url,price_confirmed_today
0,2026-09-13,A1,Duo (Net 300 + TV),300,150,No,internet_tv,3499,NaN,24,FTTH,Includes Netflix Basic,https://a1.rs/privatni/internet/opticki-internet,Yes
1,2026-09-13,A1,Duo Plus (Net 600 + TV),600,300,No,internet_tv,4499,NaN,24,FTTH,Includes Netflix Basic,https://a1.rs/privatni/internet/opticki-internet,Yes
2,2026-09-13,MTS,BOX 3 S (Net+TV+Fixed line),200,200,Yes,internet_tv_phone,4599,6099.0,24,FTTH,"Contracted price from Nov-2024 tariff update, ...",https://mts.rs/Privatni/BOX/BOX-3,Partial
3,2026-09-13,MTS,BOX 3 M (Net+TV+Fixed line),200,200,Yes,internet_tv_phone,4999,6899.0,24,FTTH,Includes HBO+ and Apollon video club; contract...,https://mts.rs/Privatni/BOX/BOX-3,Partial
4,2026-09-13,Yettel,Net TV Basic,300,150,No,internet_tv,4399,4699.0,24,FTTH,Includes 100 RSD e-bill discount; promo 2150 R...,https://www.yettel.rs/sr/privatni/tv/,Yes
5,2026-09-13,Yettel,Net TV Premium,500,250,No,internet_tv,5299,5599.0,24,FTTH,Includes 100 RSD e-bill discount; promo 2600 R...,https://www.yettel.rs/sr/privatni/tv/,Yes


In [3]:
dataframe = dataframe.drop(["contract_months", "price_rsd_no_contract", "contract_months", "symmetric", "technology", "notes", "source_url", "price_confirmed_today"], axis=1, inplace=False)
dataframe.loc[2, "download_mbps"] = 400
dataframe.loc[3, "download_mbps"] = 600
dataframe.loc[2, "upload_mbps"] = 400
dataframe.loc[3, "upload_mbps"] = 600
dataframe.loc[0, "price_rsd_month"] = 3499
dataframe.loc[1, "price_rsd_month"] = 4490
dataframe.loc[2, "price_rsd_month"] = 2700
dataframe.loc[3, "price_rsd_month"] = 3100
dataframe.loc[4, "price_rsd_month"] = 2150
dataframe.loc[5, "price_rsd_month"] = 2600
dataframe.loc[0, "on_discount"] = 0
dataframe.loc[1, "on_discount"] = 0
dataframe.loc[2, "on_discount"] = 1
dataframe.loc[3, "on_discount"] = 1
dataframe.loc[4, "on_discount"] = 1
dataframe.loc[5, "on_discount"] = 1
dataframe

,snapshot_date,provider,plan_name,download_mbps,upload_mbps,bundle_type,price_rsd_month,on_discount
0,2026-09-13,A1,Duo (Net 300 + TV),300,150,internet_tv,3499,0.0
1,2026-09-13,A1,Duo Plus (Net 600 + TV),600,300,internet_tv,4490,0.0
2,2026-09-13,MTS,BOX 3 S (Net+TV+Fixed line),400,400,internet_tv_phone,2700,1.0
3,2026-09-13,MTS,BOX 3 M (Net+TV+Fixed line),600,600,internet_tv_phone,3100,1.0
4,2026-09-13,Yettel,Net TV Basic,300,150,internet_tv,2150,1.0
5,2026-09-13,Yettel,Net TV Premium,500,250,internet_tv,2600,1.0


In [4]:
dataframe.describe()

,download_mbps,upload_mbps,price_rsd_month,on_discount
count,6.000000,6.000000,6.000000,6.000000
mean,450.000000,308.333333,3089.833333,0.666667
std,137.840488,171.512876,825.006768,0.516398
min,300.000000,150.000000,2150.000000,0.000000
25%,325.000000,175.000000,2625.000000,0.250000
50%,450.000000,275.000000,2900.000000,1.000000
75%,575.000000,375.000000,3399.250000,1.000000
max,600.000000,600.000000,4490.000000,1.000000


In [5]:
dataframe.info()

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   snapshot_date    6 non-null      str    
 1   provider         6 non-null      str    
 2   plan_name        6 non-null      str    
 3   download_mbps    6 non-null      int64  
 4   upload_mbps      6 non-null      int64  
 5   bundle_type      6 non-null      str    
 6   price_rsd_month  6 non-null      int64  
 7   on_discount      6 non-null      float64
dtypes: float64(1), int64(3), str(4)
memory usage: 516.0 bytes


In [6]:
train_dataframe = dataframe[["download_mbps", "upload_mbps", "bundle_type", "price_rsd_month", "on_discount"]]
train_dataframe

,download_mbps,upload_mbps,bundle_type,price_rsd_month,on_discount
0,300,150,internet_tv,3499,0.0
1,600,300,internet_tv,4490,0.0
2,400,400,internet_tv_phone,2700,1.0
3,600,600,internet_tv_phone,3100,1.0
4,300,150,internet_tv,2150,1.0
5,500,250,internet_tv,2600,1.0


In [7]:
label_encoder = sklearn.preprocessing.LabelEncoder()
label_encoder

LabelEncoder()

In [8]:
train_dataframe["bundle_type"] = label_encoder.fit_transform(train_dataframe["bundle_type"])
train_dataframe

,download_mbps,upload_mbps,bundle_type,price_rsd_month,on_discount
0,300,150,0,3499,0.0
1,600,300,0,4490,0.0
2,400,400,1,2700,1.0
3,600,600,1,3100,1.0
4,300,150,0,2150,1.0
5,500,250,0,2600,1.0


In [9]:
scaler = sklearn.preprocessing.StandardScaler()
scaler

,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


In [10]:
train_dataframe_norm = scaler.fit_transform(train_dataframe)
train_dataframe_norm

array([[-1.19207912, -1.01126796, -0.70710678,  0.54329206, -1.41421356],
       [ 1.19207912, -0.05322463, -0.70710678,  1.85914322, -1.41421356],
       [-0.39735971,  0.58547092,  1.41421356, -0.51762123,  0.70710678],
       [ 1.19207912,  1.86286203,  1.41421356,  0.01349931,  0.70710678],
       [-1.19207912, -1.01126796, -0.70710678, -1.24791199,  0.70710678],
       [ 0.39735971, -0.37257241, -0.70710678, -0.65040137,  0.70710678]])

In [11]:
train_dataframe = pandas.DataFrame(train_dataframe_norm, columns=train_dataframe.columns, index=train_dataframe.index)
train_dataframe

,download_mbps,upload_mbps,bundle_type,price_rsd_month,on_discount
0,-1.192079,-1.011268,-0.707107,0.543292,-1.414214
1,1.192079,-0.053225,-0.707107,1.859143,-1.414214
2,-0.397360,0.585471,1.414214,-0.517621,0.707107
3,1.192079,1.862862,1.414214,0.013499,0.707107
4,-1.192079,-1.011268,-0.707107,-1.247912,0.707107
5,0.397360,-0.372572,-0.707107,-0.650401,0.707107


In [12]:
model = sklearn.neighbors.NearestNeighbors(n_neighbors=1)
model

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",1
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None


In [13]:
model.fit(train_dataframe)
model

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",1
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'auto'
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'minkowski'
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'eu...an'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [14]:
user_wish = pandas.DataFrame([{
    'download_mbps': 500,
    'upload_mbps': 500,
    'bundle_type': 1,
    'price_rsd_month': 2000, 
    'on_discount': 1
}])
user_wish

,download_mbps,upload_mbps,bundle_type,price_rsd_month,on_discount
0,500,500,1,2000,1


In [15]:
user_wish = pandas.DataFrame(scaler.transform(user_wish), columns=user_wish.columns, index=user_wish.index)
user_wish

,download_mbps,upload_mbps,bundle_type,price_rsd_month,on_discount
0,0.39736,1.224166,1.414214,-1.447082,0.707107


In [16]:
dataframe.iloc[model.kneighbors(user_wish, return_distance=False).tolist()[0][0]]

snapshot_date                       2026-09-13
provider                                   MTS
plan_name          BOX 3 S (Net+TV+Fixed line)
download_mbps                              400
upload_mbps                                400
bundle_type                  internet_tv_phone
price_rsd_month                           2700
on_discount                                1.0
Name: 2, dtype: object

In [17]:
dataframe.to_csv("data/wifi_tv_plan.csv")

In [18]:
pandas.read_csv("data/wifi_tv_plan.csv").drop(["Unnamed: 0"], axis=1, inplace=False)

,snapshot_date,provider,plan_name,download_mbps,upload_mbps,bundle_type,price_rsd_month,on_discount
0,2026-09-13,A1,Duo (Net 300 + TV),300,150,internet_tv,3499,0.0
1,2026-09-13,A1,Duo Plus (Net 600 + TV),600,300,internet_tv,4490,0.0
2,2026-09-13,MTS,BOX 3 S (Net+TV+Fixed line),400,400,internet_tv_phone,2700,1.0
3,2026-09-13,MTS,BOX 3 M (Net+TV+Fixed line),600,600,internet_tv_phone,3100,1.0
4,2026-09-13,Yettel,Net TV Basic,300,150,internet_tv,2150,1.0
5,2026-09-13,Yettel,Net TV Premium,500,250,internet_tv,2600,1.0


In [19]:
with open("models/wifi_tv_plan_recomendation_model.pkl", "wb+") as file:
    pickle.dump(model, file)